# Machine Learning Modeling

This phase focuses on training predictive models using the engineered dataset from Phase 7.

The objective is to build machine learning models capable of predicting whether a shipment will be delivered late based on logistics attributes.

Key tasks include:

• Selecting appropriate machine learning algorithms  
• Training models using prepared datasets  
• Generating predictions using the test dataset  
• Evaluating model performance using classification metrics  
• Saving model outputs for documentation and analysis

## 1. Data Loading

The feature-engineered dataset generated in Phase 7 is loaded for machine learning model training and evaluation.

In [1]:
import pandas as pd
from pathlib import Path

BASE = Path.cwd()

# Support running from the project root or notebooks/ folder
if not (BASE / "data").is_dir():
    if (BASE.parent / "data").is_dir():
        BASE = BASE.parent
    else:
        raise FileNotFoundError(
            "Project data folder not found. "
            "Run this notebook from the project root or notebooks folder."
        )

RAW = BASE / "data" / "raw"
PROC = BASE / "data" / "processed"
DB = BASE / "data" / "logistics.db"

# Create only the directory needed for processed datasets
PROC.mkdir(parents=True, exist_ok=True)

print("Project root:", BASE)
print("Raw data:", RAW)
print("Processed data:", PROC)


data = pd.read_csv(PROC / "model_ready_dataset.csv")

data.head()

Project root: d:\Athar\LogiTech_Logistics_Analysis
Raw data: d:\Athar\LogiTech_Logistics_Analysis\data\raw
Processed data: d:\Athar\LogiTech_Logistics_Analysis\data\processed


,Warehouse_ID,Delivery_Time_Days,Weight,Shipment_Status,Delivery_Mode,Shipment_Cost,Delivery_Accuracy_Flag,Damage_Flag,Cost_Per_Weight,Late_Delivery,Carrier_Name_V444_1,Carrier_Name_V44_3
0,PLANT03,-1.344872,-0.252458,Delivered,AIR,-0.033860,0.145873,-0.166304,-0.082586,0,True,False
1,PLANT03,-1.344872,-0.228222,Delivered,AIR,-0.033860,0.145873,-0.166304,-0.084655,0,True,False
2,PLANT03,-1.344872,-0.221236,Delivered,AIR,-0.034783,0.145873,-0.166304,-0.085768,0,True,False
3,PLANT03,-1.344872,-0.209301,Delivered,AIR,-0.012242,0.145873,-0.166304,-0.070124,0,True,False
4,PLANT03,-1.344872,-0.009036,Delivered,AIR,0.103057,0.145873,-0.166304,-0.057639,0,True,False


## 2. Feature and Target Preparation

The dataset is separated into:

• **Features (X)** — input variables used by the model  
• **Target (y)** — the variable we want to predict (Late_Delivery)

Categorical variables are encoded to ensure compatibility with machine learning algorithms.

In [2]:
X = data.drop(["Late_Delivery", "Delivery_Time_Days"], axis=1)
y = data["Late_Delivery"]

X = pd.get_dummies(X, drop_first=True)

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (9215, 15)
Target shape: (9215,)


## 3. Train-Test Split

The dataset is divided into training and testing subsets.

• **Training data (80%)** is used to train the model  
• **Testing data (20%)** is used to evaluate model performance

This ensures that the model is evaluated on unseen data.

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 7372
Testing samples: 1843


## 4. Algorithm Selection

Since the target variable **Late_Delivery** is binary (0 or 1), the problem is treated as a **classification task**.

Logistic Regression is selected as the baseline model due to its simplicity, efficiency, and interpretability.

In [4]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

## 5. Model Prediction

The trained model is used to generate predictions on the test dataset.

These predictions will later be compared with actual values to evaluate the model's performance.

In [5]:
y_pred = model.predict(X_test)

## 6. Multiple Linear Regression
What factors affect shipment cost the most?

In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

warehouse_df = pd.read_csv(PROC / "warehouse_cleaned.csv")

merged_df = pd.merge(data, warehouse_df, on = 'Warehouse_ID', how = 'left')

X_reg = merged_df[["Weight", "Delivery_Time_Days", "Warehouse_Utilization_%"]]
y_reg = merged_df["Shipment_Cost"]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

lr = LinearRegression()
lr.fit(X_train_reg, y_train_reg)

y_pred_reg = lr.predict(X_test_reg)

r2 = r2_score(y_test_reg, y_pred_reg)
print("R2 Score:", round(r2, 4))

coeff_df = pd.DataFrame({
    "Feature": X_reg.columns,
    "Impact": lr.coef_
})

print(coeff_df)

R2 Score: -46.0877
                   Feature    Impact
0                   Weight  0.472552
1       Delivery_Time_Days -0.138143
2  Warehouse_Utilization_%  0.005096


## 7. Saving Model Predictions

Model predictions are saved to a file for documentation and future analysis.

In [7]:
clf_results = pd.DataFrame({
    "Actual_Delivery": y_test.values,
    "Predicted_Delivery": y_pred
})

reg_results = pd.DataFrame({
    "Actual_Cost": y_test_reg.values,
    "Predicted_Cost": y_pred_reg
})

clf_results.reset_index(drop=True, inplace=True)
reg_results.reset_index(drop=True, inplace=True)

final_results = pd.concat([clf_results, reg_results], axis=1)

final_results.to_csv(PROC / "final_model_results.csv", index=False)

print("Both model results saved successfully.")

Both model results saved successfully.


In [8]:
display(clf_results.head())
display(reg_results.head())

,Actual_Delivery,Predicted_Delivery
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0


,Actual_Cost,Predicted_Cost
0,-0.028343,-0.130549
1,-0.034099,0.090124
2,-0.028343,-0.144932
3,-0.028343,-0.142668
4,-0.028343,-0.173167


## Model Training Summary

The model training phase developed predictive models to understand delivery delays and shipment cost patterns using the prepared dataset.

Key outcomes include:

• Models were trained to identify shipments that are likely to be delayed and to analyze factors influencing shipment cost.

• Logistic Regression was used to predict delayed shipments and showed high overall accuracy, successfully identifying most on-time deliveries.

• However, delayed shipments were harder to detect, highlighting the need for better data or additional factors to improve prediction.

• Multiple Linear Regression was used to analyze shipment cost drivers, but the model showed very low accuracy, indicating that shipment cost depends on additional factors not included in the current data.

• Overall, the models provide useful insights for delivery performance, while shipment cost prediction requires further improvement with more comprehensive data.